## Topic: Tools Calling and Execution

### Agenda
- 1. Introduction of Tool Calling

- 2. Tools VS Tools Binding VS Tools Calling

- 3. Practical Example of Tool Calling

- 4. Tools Execution

- 6. Complete Example 


### 1. Introduction of Tool Calling

- Definition:
    - Tool calling is the process where the LLM(Language model) deciders, during a conversation or task, that it needs to use a specific tool(function) and generates a structured output with
        - the name of the tool.
        - and the arguments to call it with.
    
- The LLM does not actually run the tool, it just suggests the tool and the input arguments, the actual execution is handled by LangChain or the programmer.

- Key note:
    - LLM suggest the which tool is best for generate output
    - LLM can't execute the tool.

In [ ]:
""" 
    - Tool Calling Flow
    ========================
User Question
      │
      ▼
     LLM
      │
      │ Decides a tool is needed
      ▼
  Tool Call
      │
      │ tool name + arguments
      ▼
 Tool Execution
      │
      ▼
 Tool Result
      │
      ▼
     LLM
      │
      ▼
 Final Answer



"""

### 2. Tools VS Tools Binding VS Tools Calling

- 1. Tool
    - A tool provides a additional capability of LLM.

- 2. Tool Binding
    - Binding makes the tool available to the LLM.

- 3. Tool Calling
    - The model actually requests the tool.

### 3. Practical Example of Tool Calling

In [1]:
# Practical Example — Simple Tool Calling

# Step 1 — Create a Tool
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


# Step 2 — Create the LLM
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load variables from .env
load_dotenv()

# Create the chat model
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Step 3 — Bind the Tool
model_with_tools = model.bind_tools(
    [multiply]
)

# Ask the Model a Question
response = model_with_tools.invoke(
    "What is 25 multiplied by 10?"
)

print(response)

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


content='' additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_3c050432-fa02-476f-b251-464763acb88e', 'function': {'arguments': '{"a":25,"b":10}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 129, 'total_tokens': 164, 'completion_time': 0.060611787, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006283557, 'prompt_tokens_details': None, 'queue_time': 0.208574821, 'total_time': 0.066895344}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0b3ad-e59f-74e1-851d-1a2c97d37795-0' tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 10}, 'id': 'fc_3c050432-fa02-476f-b251-464763acb88e', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 129, 'output_tokens': 35, 'total_to

In [3]:
# Execute the requested tool
tool_call = response.tool_calls[0]
tool_call

{'name': 'multiply',
 'args': {'a': 25, 'b': 10},
 'id': 'fc_3c050432-fa02-476f-b251-464763acb88e',
 'type': 'tool_call'}

In [4]:
# when don't requested the tools to execute
response_no_tool_call = model_with_tools.invoke("Hello! How are you?")

print(response_no_tool_call)

content='Hello! I’m doing great—thanks for asking. How can I help you today?' additional_kwargs={'reasoning_content': 'User says "Hello! How are you?" We respond politely.'} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 126, 'total_tokens': 167, 'completion_time': 0.044326195, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007077828, 'prompt_tokens_details': None, 'queue_time': 0.308676819, 'total_time': 0.051404023}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3cdd24b83b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0b3b0-f771-7960-99a9-7ab5c54cb560-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 126, 'output_tokens': 41, 'total_tokens': 167, 'output_token_details': {'reasoning': 14}}


### 4. Tools Execution
- Definition:
    - Tool Execution is the step where the actual python function(tool) is running the input arguments that the LLM suggested during tool calling.

- In simple words:
    - The LLM say:
        - Hey call the multiply tool with a= 25, b = 10

    - Tool Execution is when we or LangChain actually run.
    - multiply(a=25, b=10)
    - and get the result = 250

- Syntax:
    - suggested_tool_function_name.invoke(args)

In [5]:
# Execute the requested tool
tool_call = response.tool_calls[0]
tool_call

{'name': 'multiply',
 'args': {'a': 25, 'b': 10},
 'id': 'fc_3c050432-fa02-476f-b251-464763acb88e',
 'type': 'tool_call'}

In [7]:
# We can execute the requested tool manually
result = multiply.invoke(
    {'a': 25, 'b': 10}
)

print(f"Result: {result}")

Result: 250


In [8]:
# We can execute the requested tool manually another way
result = multiply.invoke(
    tool_call["args"]    
)

print(f"Result: {result}")

Result: 250


In [9]:
# if we want to see the Tools message
tool_message = multiply.invoke(tool_call)
tool_message

ToolMessage(content='250', name='multiply', tool_call_id='fc_3c050432-fa02-476f-b251-464763acb88e')

- Key Note:
    - user -> LLm -> tool call -> tool result 

### Example 2

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool


# --------------------------------------------------
# 1. Load environment variables
# --------------------------------------------------

load_dotenv()


# --------------------------------------------------
# 2. Create a custom tool
# --------------------------------------------------

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


# --------------------------------------------------
# 3. Create the LLM
# --------------------------------------------------

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# --------------------------------------------------
# 4. Bind the tool to the model
# --------------------------------------------------

model_with_tools = model.bind_tools(
    [multiply]
)


# --------------------------------------------------
# 5. Ask the model a question
# --------------------------------------------------

response = model_with_tools.invoke(
    "What is 25 multiplied by 10?"
)


# --------------------------------------------------
# 6. Inspect the tool call
# --------------------------------------------------

print("Tool Call:")
print(response.tool_calls)


# --------------------------------------------------
# 7. Execute the requested tool
# --------------------------------------------------

tool_call = response.tool_calls[0]

result = multiply.invoke(
    tool_call["args"]
)


# --------------------------------------------------
# 8. Display the tool result
# --------------------------------------------------

print("\nTool Result:")
print(result)

### 5. Complete Example
- user -> LLm -> tool call -> tool result-> LLM -> Final Result

In [ ]:
""" 
                  ┌──────────────┐
                  │     User     │
                  └──────┬───────┘
                         │
                         ▼
                  ┌──────────────┐
                  │     LLM      │
                  └──────┬───────┘
                         │
                    Tool Call
                         │
                         ▼
                  ┌──────────────┐
                  │     Tool     │
                  └──────┬───────┘
                         │
                    Tool Result
                         │
                         ▼
                  ┌──────────────┐
                  │     LLM      │
                  └──────┬───────┘
                         │
                         ▼
                  Final Answer

"""

In [ ]:
# import libraries
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


In [11]:
# --------------------------------------------------
# 1. Load environment variables
# --------------------------------------------------

load_dotenv()


# --------------------------------------------------
# 2. Create a custom tool
# --------------------------------------------------

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b



In [12]:
# --------------------------------------------------
# 3. Create the LLM
# --------------------------------------------------

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# --------------------------------------------------
# 4. Bind the tool to the model
# --------------------------------------------------
model_with_tools = model.bind_tools(
    [multiply]
)


In [13]:
# --------------------------------------------------
# 5. Ask the model a question
# --------------------------------------------------

query = HumanMessage("what is Product of 10 by 10")

message = [query]

message

[HumanMessage(content='what is Product of 10 by 10', additional_kwargs={}, response_metadata={})]

In [ ]:
# Call the LLm and get the LLM suggest
response = model_with_tools.invoke(message)
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what is Product of 10 by 10". We should use the multiply function.', 'tool_calls': [{'id': 'fc_fb35f39c-fb73-4c0d-9188-7638e6c24528', 'function': {'arguments': '{"a":10,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 129, 'total_tokens': 177, 'completion_time': 0.050880147, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.007367678, 'prompt_tokens_details': None, 'queue_time': 0.315994789, 'total_time': 0.058247825}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b3ca-ea97-7d70-adce-741c1ef76e35-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 10}, 'id': 'fc_fb35f39c-fb73-4c0d-9188-7638e6c24528', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

In [18]:
# add llm suggestion in message
message.append(response)

In [19]:
message

[HumanMessage(content='what is Product of 10 by 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what is Product of 10 by 10". We should use the multiply function.', 'tool_calls': [{'id': 'fc_fb35f39c-fb73-4c0d-9188-7638e6c24528', 'function': {'arguments': '{"a":10,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 129, 'total_tokens': 177, 'completion_time': 0.050880147, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.007367678, 'prompt_tokens_details': None, 'queue_time': 0.315994789, 'total_time': 0.058247825}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b3ca-ea97-7d70-adce-741c1ef76e35-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 10}, 'id':

In [20]:

# --------------------------------------------------
# 6. Inspect the tool call
# --------------------------------------------------

print("Tool Call:")
print(response.tool_calls)


Tool Call:
[{'name': 'multiply', 'args': {'a': 10, 'b': 10}, 'id': 'fc_fb35f39c-fb73-4c0d-9188-7638e6c24528', 'type': 'tool_call'}]


In [ ]:
# --------------------------------------------------
# 7. Execute the requested tool
# --------------------------------------------------

tool_call = response.tool_calls[0]


In [23]:
# if we want to see the Tools message
tool_message = multiply.invoke(tool_call)
tool_message

ToolMessage(content='100', name='multiply', tool_call_id='fc_fb35f39c-fb73-4c0d-9188-7638e6c24528')

In [24]:
# append the tool message
message.append(tool_message)

In [25]:
message

[HumanMessage(content='what is Product of 10 by 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what is Product of 10 by 10". We should use the multiply function.', 'tool_calls': [{'id': 'fc_fb35f39c-fb73-4c0d-9188-7638e6c24528', 'function': {'arguments': '{"a":10,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 129, 'total_tokens': 177, 'completion_time': 0.050880147, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.007367678, 'prompt_tokens_details': None, 'queue_time': 0.315994789, 'total_time': 0.058247825}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b3ca-ea97-7d70-adce-741c1ef76e35-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 10}, 'id':

In [27]:
# pass the everything to LLm
final_response = model_with_tools.invoke(message)

print(f"Final Result \n")

print(final_response)

Final Result 

content='The product of 10 by 10 is **100**.' additional_kwargs={'reasoning_content': 'The user asked: "what is Product of 10 by 10". We responded with the multiplication tool. The tool returned 100. We should respond with the answer.'} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 204, 'total_tokens': 262, 'completion_time': 0.0593269, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.009856339, 'prompt_tokens_details': None, 'queue_time': 0.400453108, 'total_time': 0.069183239}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0b3d3-47b9-7b90-a47f-d324d42c290d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 204, 'output_tokens': 58, 'total_tokens': 262, 'output_token_details': {'reasoning': 36}}


In [28]:
print(final_response.content)

The product of 10 by 10 is **100**.
